# PeakVI 官方基线

本 notebook 使用固定发布版 `scvi-tools==1.4.2` 的 `scvi.model.PEAKVI`。训练输入只使用真实训练集按区域中位数填补后的完整矩阵；模型结构、重建损失、优化器、训练划分及训练参数均沿用官方默认。默认最多 500 轮并启用官方早停，只允许设置随机种子、设备和路径。

生成目标的细胞数、`Subclass` 和行顺序来自目标文件的 `obs`；不读取目标矩阵 `X`。每个目标细胞从同 `Subclass` 的训练细胞中有放回抽取一个供体，再调用官方随机后验接口生成全部区域的概率。最后仅进行 `clip(0, 1)` 和 float32 存储，不加缺失掩码、不对齐分布、不复制区域、不修改损失。

**方法解释边界：** PeakVI 为 scATAC 可及性建模，官方重建目标为 `(x > 0)`，因此不会拟合甲基化比例的连续数值分布。本适配保留填补比例原值供官方编码器使用，并保留该二元重建目标；官方可能提示输入不是整数计数，该提示不被隐藏。`Subclass` 只用于选择供体，模型本身保持默认的无监督训练。

官方 `PEAKVAE` 没有可用的 `sample` 实现，因此这里使用 `get_normalized_accessibility(indices=..., use_z_mean=False, return_numpy=True)`。所得矩阵是从训练供体随机后验解码的可及性概率，不能解释为先验生成样本、伯努利计数或甲基化比例预测。归一化、阈值和区域筛选参数均保持默认。

参考：[官方 API](https://docs.scvi-tools.org/en/stable/api/reference/scvi.model.PEAKVI.html)、[固定版本模型源码](https://github.com/scverse/scvi-tools/blob/1.4.2/src/scvi/model/_peakvi.py)、[固定版本模块与损失](https://github.com/scverse/scvi-tools/blob/1.4.2/src/scvi/module/_peakvae.py)。


## 运行条件与路径

选择服务器的 `simscDNAm` 内核，在 `/home/sccasimp/program/xb/simscDNAm` 工作区运行。入口通过 `BASELINE_TRAINING`、`BASELINE_TARGET`、`BASELINE_OUTPUT`、`BASELINE_SEED`、`BASELINE_DEVICE` 提供设置；也可在末尾执行单元手动设置对应环境变量。

服务器所有填补缓存、模型、供体表、生成结果及核验报告都写入 `/home/sccasimp/data/xb/outputs/simscDNAm`，路径由项目统一的 `check_output` 检查。正式训练保留约 27–31 万个区域，默认网络会随区域数增大，需在服务器检查内存和显存后运行。本 notebook 不自动安装依赖，也不缩小网络或减少训练轮数。


In [ ]:
import json
import os
from importlib.metadata import version
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd

# 支持从工作区根目录或 notebooks 目录启动。
PROJECT = Path.cwd().resolve()
if not (PROJECT / "data/baseline.py").is_file() and (PROJECT.parent / "data/baseline.py").is_file():
    PROJECT = PROJECT.parent
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from data.baseline import prepare_imputed_training, clip_generated_matrix
from data.paths import OUTPUTS, check_output, require_environment


## 官方训练与随机后验生成

以下函数只包装官方调用。训练文件同级的 `imputed_baselines/` 缓存由共用预处理函数生成，模型保存到 `model/`；`donor_cells.csv` 逐行记录同类别的训练供体，便于复核生成来源。目标观测标签来自共用元数据读取函数，适配器只直接读取填补后的训练文件。


In [ ]:
def run_peakvi(training_path, target_path, output_dir, seed=42, device="auto"):
    require_environment()
    import scvi

    installed_version = version("scvi-tools")
    if installed_version != "1.4.2":
        raise RuntimeError(f"本基线固定 scvi-tools==1.4.2，当前为 {installed_version}")
    training_path, target_path = Path(training_path), Path(target_path)
    output_dir = check_output(output_dir)
    output_path = output_dir / "data.h5ad"
    if output_path.resolve() in {training_path.resolve(), target_path.resolve()}:
        raise ValueError("输出文件不能覆盖训练或目标输入")

    # 仅映射设备；不向官方 train 传入其它超参数。
    if device == "auto":
        device_kwargs = {}
    elif device == "cpu":
        device_kwargs = {"accelerator": "cpu", "devices": 1}
    elif device == "cuda" or device.startswith("cuda:"):
        index = 0 if device == "cuda" else int(device.split(":", 1)[1])
        if index < 0:
            raise ValueError("CUDA 设备索引必须非负")
        device_kwargs = {"accelerator": "gpu", "devices": [index]}
    else:
        raise ValueError("设备仅支持 auto、cpu、cuda 或 cuda:索引")

    output_dir.mkdir(parents=True, exist_ok=True)
    filled_path, target_obs, imputation_report = prepare_imputed_training(
        training_path, target_path, training_path.parent / "imputed_baselines"
    )
    training = ad.read_h5ad(filled_path)
    training_labels = training.obs["Subclass"].astype(str).to_numpy()
    target_labels = target_obs["Subclass"].astype(str).to_numpy()
    unknown = set(target_labels) - set(training_labels)
    if not len(target_labels) or unknown:
        raise ValueError(f"目标为空或含训练集中不存在的类别：{sorted(unknown)}")

    scvi.settings.seed = int(seed)
    scvi.settings.logging_dir = output_dir / "training_logs"
    scvi.model.PEAKVI.setup_anndata(training)
    model = scvi.model.PEAKVI(training)
    model.train(**device_kwargs)
    model.save(str(output_dir / "model"), overwrite=True)

    # 每个重复供体仍由官方接口独立抽取后验潜变量，不使用后验均值。
    rng = np.random.default_rng(seed)
    donor_indices = np.empty(len(target_labels), dtype=np.int64)
    for label in pd.unique(target_labels):
        target_rows = np.flatnonzero(target_labels == label)
        available = np.flatnonzero(training_labels == label)
        donor_indices[target_rows] = rng.choice(available, size=len(target_rows), replace=True)
    raw = model.get_normalized_accessibility(
        indices=donor_indices, use_z_mean=False, return_numpy=True
    )
    if raw.shape != (len(target_obs), training.n_vars):
        raise ValueError("官方输出没有保留目标细胞数和完整区域轴")
    generated, clipping_report = clip_generated_matrix(raw)
    synthetic = ad.AnnData(X=generated, obs=target_obs.copy(), var=training.var.copy())
    synthetic.obs["origin"] = "Synthetic"
    synthetic.write_h5ad(output_path, compression="gzip")

    pd.DataFrame({
        "目标细胞": target_obs.index.astype(str),
        "Subclass": target_labels,
        "训练行索引": donor_indices,
        "训练供体细胞": training.obs_names[donor_indices].astype(str),
    }).to_csv(output_dir / "donor_cells.csv", index=False, encoding="utf-8-sig")
    report = {
        "方法": "PeakVI",
        "scvi-tools版本": installed_version,
        "随机种子": int(seed),
        "官方模型参数": "默认",
        "训练参数": "官方默认：max_epochs=500，early_stopping=True；仅设备参数可指定",
        "训练设备参数": device_kwargs,
        "训练输入": str(filled_path),
        "训练形状": list(training.shape),
        "输出形状": list(synthetic.shape),
        "生成方式": "按 Subclass 抽取训练供体，随机后验解码得到可及性概率",
        "目标矩阵读取": False,
        "类别数量": {str(label): int(count) for label, count in target_obs["Subclass"].value_counts().items()},
        "训练填补": imputation_report,
        "生成后截断": clipping_report,
        "输出文件": str(output_path),
    }
    (output_dir / "verification.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps({"方法": "PeakVI", "输出形状": list(synthetic.shape), "输出文件": str(output_path)}, ensure_ascii=False))
    return output_path


## 执行并核验

正式执行会训练官方默认模型。完成后检查 `data.h5ad` 的形状、`Subclass`、区域名称及有限范围，并检查 `donor_cells.csv` 的供体类别。`verification.json` 记录包版本、填补来源、概率输出定义和截断数量。模型保存不重复嵌入训练数据，复载时须传入同一份 `imputed_baselines/` 填补训练文件。

本地接口检查用小型替身隔离昂贵的官方训练，验证默认调用、同类供体、目标元数据、完整区域轴及仅截断行为；它不代表正式数据训练已完成，也不代表模型质量已经评估。


In [ ]:
required = ["BASELINE_TRAINING", "BASELINE_TARGET"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError(f"请先设置输入环境变量：{missing}")
OUTPUT_PATH = run_peakvi(
    training_path=os.environ["BASELINE_TRAINING"],
    target_path=os.environ["BASELINE_TARGET"],
    output_dir=os.environ.get("BASELINE_OUTPUT", str(OUTPUTS / "baseline/PeakVI")),
    seed=int(os.environ.get("BASELINE_SEED", "42")),
    device=os.environ.get("BASELINE_DEVICE", "auto"),
)
